# Data Merging and Analysis Notebook

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta

# CONFIGURATION
CONFIG = {
    'DATA_DIR': Path('../data'),
    'OUTPUT_DIR': Path('../data')
}

def load_data(file_path):
    """Load CSV data with error handling."""
    try:
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        df = pd.read_csv(file_path)
        if df.empty:
            raise pd.errors.EmptyDataError(f"The file is empty: {file_path}")
        return df
    except Exception as e:
        print(f"Error loading {file_path.name}: {e}")
        return None

## 1. Load Data

In [7]:
customers_clean = load_data(CONFIG['DATA_DIR'] / 'customers_clean.csv')
orders_clean = load_data(CONFIG['DATA_DIR'] / 'orders_clean.csv')
products = load_data(CONFIG['DATA_DIR'] / 'products.csv')

if customers_clean is not None and orders_clean is not None and products is not None:
    print("All datasets loaded successfully!")
    # Ensure dates are parsed correctly after loading
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'])
    orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'])

All datasets loaded successfully!


## 2. Merging
### 2.1 Join Orders onto Customers
Name result: `orders_with_customers`

In [12]:
orders_with_customers = customers_clean.merge(orders_clean, on='customer_id', how='left')
print(f"Orders with Customers Shape: {orders_with_customers.shape}")
orders_with_customers.head()

Orders with Customers Shape: (25, 12)


,customer_id,name,email,region,signup_date,is_valid_email,order_id,product,amount,order_date,status,order_year_month
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1012,P002,814.51,2023-03-14,pending,2023-03
1,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1013,P005,928.51,2023-10-01,cancelled,2023-10
2,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1025,P005,1082.50,2023-03-17,completed,2023-03
3,C002,Bob Smith,bob@test.com,South,2023-02-20,True,OR1006,P003,841.31,2023-01-30,completed,2023-01
4,C002,Bob Smith,bob@test.com,South,2023-02-20,True,OR1019,P005,957.99,2023-04-23,cancelled,2023-04


### 2.2 Join Products onto Orders
Name result: `full_data`

In [14]:
full_data = orders_with_customers.merge(products, left_on='product', right_on='product_id', how='left')
print(f"Full Data Shape: {full_data.shape}")
full_data.head()

Full Data Shape: (25, 16)


,customer_id,name,email,region,signup_date,is_valid_email,order_id,product,amount,order_date,status,order_year_month,product_id,product_name,category,unit_price
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1012,P002,814.51,2023-03-14,pending,2023-03,P002,Smartphone,Electronics,800.0
1,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1013,P005,928.51,2023-10-01,cancelled,2023-10,P005,Monitor,Electronics,300.0
2,C001,Alice Johnson,alice@example.com,North,2023-01-15,True,OR1025,P005,1082.50,2023-03-17,completed,2023-03,P005,Monitor,Electronics,300.0
3,C002,Bob Smith,bob@test.com,South,2023-02-20,True,OR1006,P003,841.31,2023-01-30,completed,2023-01,P003,Coffee Maker,Appliances,50.0
4,C002,Bob Smith,bob@test.com,South,2023-02-20,True,OR1019,P005,957.99,2023-04-23,cancelled,2023-04,P005,Monitor,Electronics,300.0


### 2.3 Report Mismatches

In [15]:
missing_customers = orders_clean[~orders_clean['customer_id'].isin(customers_clean['customer_id'])]
print(f"Order rows with no matching customer: {len(missing_customers)}")

missing_products = full_data[full_data['order_id'].notna() & full_data['product_id'].isna()]
print(f"Order rows with no matching product: {len(missing_products)}")

Order rows with no matching customer: 0
Order rows with no matching product: 0


## 3. Analysis Tasks

### 3.1 Monthly Revenue Trend
Compute total completed order revenue grouped by `order_year_month`.

In [16]:
monthly_revenue = full_data[full_data['status'] == 'completed'] \
    .groupby('order_year_month')['amount'] \
    .sum() \
    .reset_index() \
    .rename(columns={'amount': 'total_revenue'})

monthly_revenue.to_csv(CONFIG['OUTPUT_DIR'] / 'monthly_revenue.csv', index=False)
monthly_revenue

,order_year_month,total_revenue
0,2023-01,841.31
1,2023-03,1082.50
2,2023-04,1351.78
3,2023-06,2211.66
4,2023-07,2272.02
5,2023-09,2135.03


### 3.2 Top Customers
Identify top 10 customers by total spend (completed orders only).

In [19]:
top_customers = full_data[full_data['status'] == 'completed'] \
    .groupby(['customer_id', 'name', 'region'])['amount'] \
    .sum() \
    .reset_index() \
    .rename(columns={'amount': 'total_spend'}) \
    .sort_values(by='total_spend', ascending=False) \
    .head(10)

top_customers

,customer_id,name,region,total_spend
4,C005,Eva Garcia,Central,3261.47
2,C003,Charlie Brown,East,1520.42
6,C007,Grace Lee,South,1351.78
3,C004,David Wilson,West,1152.35
0,C001,Alice Johnson,North,1082.50
1,C002,Bob Smith,South,841.31
5,C006,Frank Miller,North,684.47


### 3.3 Category Performance
Group by product category, compute revenue, avg order value, and count.

In [18]:
category_performance = full_data[full_data['status'] == 'completed'] \
    .groupby('category')['amount'] \
    .agg(['sum', 'mean', 'count']) \
    .reset_index() \
    .rename(columns={'sum': 'total_revenue', 'mean': 'avg_order_value', 'count': 'order_count'})

category_performance.to_csv(CONFIG['OUTPUT_DIR'] / 'category_performance.csv', index=False)
category_performance

,category,total_revenue,avg_order_value,order_count
0,Appliances,841.31,841.310000,1
1,Electronics,6985.15,997.878571,7
2,Furniture,2067.84,1033.920000,2


### 3.4 Regional Analysis
Group by region, compute customers, orders, revenue, and revenue per customer.

In [20]:
regional_analysis = full_data[full_data['status'] == 'completed'] \
    .groupby('region') \
    .agg({
        'customer_id': 'nunique', 
        'order_id': 'count', 
        'amount': 'sum'
    }) \
    .reset_index() \
    .rename(columns={'customer_id': 'customer_count', 'order_id': 'order_count', 'amount': 'total_revenue'})

regional_analysis['revenue_per_customer'] = regional_analysis['total_revenue'] / regional_analysis['customer_count']
regional_analysis.to_csv(CONFIG['OUTPUT_DIR'] / 'regional_analysis.csv', index=False)
regional_analysis

,region,customer_count,order_count,total_revenue,revenue_per_customer
0,Central,1,3,3261.47,3261.470
1,East,1,2,1520.42,1520.420
2,North,2,2,1766.97,883.485
3,South,2,2,2193.09,1096.545
4,West,1,1,1152.35,1152.350


### 3.5 Churn Indicator
Flag customers with no completed orders in the past 90 days.

In [22]:
latest_date = full_data['order_date'].max()
churn_threshold = latest_date - timedelta(days=90)

last_order_per_customer = full_data[full_data['status'] == 'completed'] \
    .groupby('customer_id')['order_date'] \
    .max() \
    .reset_index() \
    .rename(columns={'order_date': 'last_order_date'})

last_order_per_customer['churned'] = last_order_per_customer['last_order_date'] < churn_threshold

top_customers = top_customers.merge(last_order_per_customer[['customer_id', 'churned']], on='customer_id', how='left')
top_customers.to_csv(CONFIG['OUTPUT_DIR'] / 'top_customers.csv', index=False)
top_customers

,customer_id,name,region,total_spend,churned_x,churned_y
0,C005,Eva Garcia,Central,3261.47,False,False
1,C003,Charlie Brown,East,1520.42,False,False
2,C007,Grace Lee,South,1351.78,True,True
3,C004,David Wilson,West,1152.35,False,False
4,C001,Alice Johnson,North,1082.50,True,True
5,C002,Bob Smith,South,841.31,True,True
6,C006,Frank Miller,North,684.47,False,False
